In [1]:
#cell 1
# Install runtime dependencies for GPT-OSS + vLLM on Colab / Blackwell.

!pip -q install -U uv

!uv pip install --system -U openai requests tqdm jsonschema psutil numpy pandas accelerate safetensors huggingface_hub

# Remove optional packages that may break imports or pull mismatched CUDA wheels.
!uv pip uninstall --system -y torchcodec torchvision torchaudio sentence-transformers || true

# Transformers is used for tokenizer-based input truncation.
!uv pip install --system -U "transformers>=4.56.0"

# GPT-OSS support is available in current vLLM releases.
# For this Colab GPU, first try CUDA 13 / Blackwell wheels.
!uv pip install --system -U vllm --torch-backend=cu130 --extra-index-url https://wheels.vllm.ai/nightly/cu130 || \
 uv pip install --system -U vllm --torch-backend=auto

# Optional fallback only if the lines above fail in your Colab:
# !uv pip install --system --pre -U "vllm==0.10.1+gptoss" \
#     --extra-index-url https://wheels.vllm.ai/gpt-oss/ \
#     --extra-index-url https://download.pytorch.org/whl/nightly/cu128 \
#     --index-strategy unsafe-best-match

import sys
import importlib.metadata as md

import torch
import vllm
import transformers

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)
print("transformers:", transformers.__version__)

for pkg in ["torchcodec", "torchvision", "torchaudio", "sentence-transformers"]:
    try:
        print(pkg + ":", md.version(pkg))
    except Exception:
        print(pkg + ": not installed")

!nvidia-smi

Using Python 3.12.13 environment at: /usr
Resolved 71 packages in 120ms
Prepared 8 packages in 0.32ms
Uninstalled 8 packages in 127ms
Installed 8 packages in 115ms
 - numpy==2.3.5
 + numpy==2.5.0
 - nvidia-cublas==13.1.0.3
 + nvidia-cublas==13.1.1.3
 - nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cudnn-cu13==9.20.0.48
 - nvidia-cusparselt-cu13==0.8.0
 + nvidia-cusparselt-cu13==0.8.1
 - nvidia-nccl-cu13==2.28.9
 + nvidia-nccl-cu13==2.29.7
 - setuptools==80.10.2
 + setuptools==81.0.0
 - torch==2.11.0+cu130
 + torch==2.12.1
 - triton==3.6.0
 + triton==3.7.1
Using Python 3.12.13 environment at: /usr
Uninstalled 2 packages in 49ms
 - torchaudio==2.11.0+cu130
 - torchvision==0.26.0+cu130
Using Python 3.12.13 environment at: /usr
Resolved 27 packages in 82ms
Checked 27 packages in 0.27ms
Using Python 3.12.13 environment at: /usr
Resolved 191 packages in 1.82s
Prepared 10 packages in 26ms
Uninstalled 8 packages in 109ms
Installed 10 packages in 110ms
 - numpy==2.5.0
 + numpy==2.3.5
 - nvidia-cublas=

In [2]:
#cell 2
# Imports and global configuration.

import os
import re
import gc
import json
import time
import shlex
import shutil
import psutil
import subprocess
import traceback
import site
import glob

from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from transformers import AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# GPT-OSS model.
LLM_MODEL_NAME = "openai/gpt-oss-120b"

# GPT-OSS supports low / medium / high.
# Keep this from the GPT-OSS notebook.
REASONING_EFFORT = "high"

PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# Keep the final LogicRAG prompt behavior close to the Qwen LogicRAG notebook.
MAX_MODEL_LEN = 131072
MAX_INPUT_TOKENS = 14336
MEMORY_MAX_TOKENS = 10000

# GPT-OSS completion budget includes reasoning tokens + final answer.
ANSWER_MAX_TOKENS = 32768

GPU_MEMORY_UTILIZATION = 0.95
MAX_NUM_SEQS = 1
MAX_NUM_BATCHED_TOKENS = 8192
KV_CACHE_DTYPE = "fp8"
ENABLE_PREFIX_CACHING = False
MAX_CUDAGRAPH_CAPTURE_SIZE = 2048

# Critical fix for Colab / Blackwell / CUDA 13.
USE_FLASHINFER_SAMPLER = False
DISABLE_FLASHINFER_COMPLETELY = False
ATTENTION_BACKEND = None
ADD_NVIDIA_PIP_LIBS_TO_LD_LIBRARY_PATH = True

SERVER_LOG_PATH = Path("/content/vllm_gpt_oss_logicrag_final_answer_server.log")
SERVER_PID_PATH = Path("/content/vllm_gpt_oss_logicrag_final_answer_server.pid")

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE_ROOT / "final_project"
WORK_DIR = PROJECT_DIR / "logicRAG"

# Evidence files are auto-resolved from either:
# final_project/logicRAG/evidence/*.json
# or final_project/logicRAG/*.json
EVIDENCE_DIR_CANDIDATES = [
    WORK_DIR / "evidence",
    WORK_DIR,
]

# Answer folder is auto-resolved to the folder that already contains Qwen answers.
ANSWER_DIR_CANDIDATES = [
    WORK_DIR / "answer",
    DRIVE_ROOT / "final_answer" / "logicRAG" / "answer",
]

LOCAL_RUNTIME_DIR = Path("/content/final_project_logicrag_gpt_oss_final_answer")
LOCAL_EVIDENCE_DIR = LOCAL_RUNTIME_DIR / "evidence"
LOCAL_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

DATASET_FILE_NAMES = {
    "hotpotqa": {
        "evidence_file": "hotpotqa_evidence.json",
        "qwen_answer_file": "hotpotqa_qwen3.5_answers.json",
        "gpt_oss_answer_file": "hotpotqa_gpt_oss_120b_answers.json",
    },
    "2wikimultihopqa": {
        "evidence_file": "2wikimultihopqa_evidence.json",
        "qwen_answer_file": "2wikimultihopqa_qwen3.5_answers.json",
        "gpt_oss_answer_file": "2wikimultihopqa_gpt_oss_120b_answers.json",
    },
}

DATASET_RUN_ORDER = ["hotpotqa", "2wikimultihopqa"]

EXPECTED_NUM_RECORDS_PER_DATASET = 1000

# Use None to process all records.
ANSWER_START_INDEX = 0
ANSWER_END_INDEX = None

SAVE_EVERY_N = 1
CLEAR_CACHE_EVERY_N = 25
RESUME_IF_EXISTS = True

print("Model:", LLM_MODEL_NAME)
print("Reasoning effort:", REASONING_EFFORT)
print("Max model len:", MAX_MODEL_LEN)
print("Max input tokens:", MAX_INPUT_TOKENS)
print("Memory max tokens:", MEMORY_MAX_TOKENS)
print("Answer max tokens:", ANSWER_MAX_TOKENS)
print("GPU memory utilization:", GPU_MEMORY_UTILIZATION)
print("Max num batched tokens:", MAX_NUM_BATCHED_TOKENS)
print("KV cache dtype:", KV_CACHE_DTYPE)
print("Work dir:", WORK_DIR)

Model: openai/gpt-oss-120b
Reasoning effort: high
Max model len: 131072
Max input tokens: 14336
Memory max tokens: 10000
Answer max tokens: 32768
GPU memory utilization: 0.95
Max num batched tokens: 8192
KV cache dtype: fp8
Work dir: /content/drive/MyDrive/final_project/logicRAG


In [3]:
#cell 3
# Mount Google Drive and resolve LogicRAG evidence/answer paths.

from google.colab import drive

MOUNTPOINT = Path("/content/drive")
drive.mount(str(MOUNTPOINT), force_remount=True)

assert PROJECT_DIR.exists(), f"PROJECT_DIR does not exist: {PROJECT_DIR}"
assert WORK_DIR.exists(), f"WORK_DIR does not exist: {WORK_DIR}"

def resolve_existing_file(file_name: str, candidates: List[Path]) -> Path:
    """Resolve a file from candidate directories."""
    checked = []

    for directory in candidates:
        path = directory / file_name
        checked.append(path)
        if path.exists():
            return path

    raise FileNotFoundError(
        "Could not find file:\n"
        + file_name
        + "\nChecked:\n"
        + "\n".join(str(x) for x in checked)
    )

def choose_answer_dir() -> Path:
    """Choose the answer directory that contains existing Qwen answer files when possible."""
    for directory in ANSWER_DIR_CANDIDATES:
        if all(
            (directory / DATASET_FILE_NAMES[name]["qwen_answer_file"]).exists()
            for name in DATASET_RUN_ORDER
        ):
            return directory

    for directory in ANSWER_DIR_CANDIDATES:
        if directory.exists():
            return directory

    return ANSWER_DIR_CANDIDATES[0]

def file_is_same_size(src: Path, dst: Path) -> bool:
    """Check whether the destination file exists and has the same size as the source."""
    return dst.exists() and dst.stat().st_size == src.stat().st_size

def copy_file_to_local(src: Path, dst: Path) -> None:
    """Copy a file to local disk using a temporary file to avoid partial copies."""
    dst.parent.mkdir(parents=True, exist_ok=True)

    if file_is_same_size(src, dst):
        print(f"Local copy already exists: {dst}")
        return

    tmp = dst.with_name(dst.name + ".tmp")

    if tmp.exists():
        tmp.unlink()

    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

    print(f"Copied to local disk: {src} -> {dst}")

DRIVE_ANSWER_DIR = choose_answer_dir()
DRIVE_ANSWER_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {}

for dataset_name, names in DATASET_FILE_NAMES.items():
    drive_evidence_path = resolve_existing_file(
        names["evidence_file"],
        EVIDENCE_DIR_CANDIDATES,
    )

    local_evidence_path = LOCAL_EVIDENCE_DIR / names["evidence_file"]
    answer_path = DRIVE_ANSWER_DIR / names["gpt_oss_answer_file"]

    DATASETS[dataset_name] = {
        "drive_evidence_path": drive_evidence_path,
        "local_evidence_path": local_evidence_path,
        "answer_path": answer_path,
    }

    copy_file_to_local(drive_evidence_path, local_evidence_path)

    print("=" * 100)
    print("Dataset:", dataset_name)
    print("Drive evidence:", drive_evidence_path)
    print("Local evidence:", local_evidence_path)
    print("GPT-OSS answer output:", answer_path)
    print("Local evidence size MB:", local_evidence_path.stat().st_size / (1024 ** 2))

print("=" * 100)
print("Answer directory:", DRIVE_ANSWER_DIR)

Mounted at /content/drive
Copied to local disk: /content/drive/MyDrive/final_project/logicRAG/evidence/hotpotqa_evidence.json -> /content/final_project_logicrag_gpt_oss_final_answer/evidence/hotpotqa_evidence.json
Dataset: hotpotqa
Drive evidence: /content/drive/MyDrive/final_project/logicRAG/evidence/hotpotqa_evidence.json
Local evidence: /content/final_project_logicrag_gpt_oss_final_answer/evidence/hotpotqa_evidence.json
GPT-OSS answer output: /content/drive/MyDrive/final_project/logicRAG/answer/hotpotqa_gpt_oss_120b_answers.json
Local evidence size MB: 16.1061372756958
Copied to local disk: /content/drive/MyDrive/final_project/logicRAG/evidence/2wikimultihopqa_evidence.json -> /content/final_project_logicrag_gpt_oss_final_answer/evidence/2wikimultihopqa_evidence.json
Dataset: 2wikimultihopqa
Drive evidence: /content/drive/MyDrive/final_project/logicRAG/evidence/2wikimultihopqa_evidence.json
Local evidence: /content/final_project_logicrag_gpt_oss_final_answer/evidence/2wikimultihopqa

In [4]:
#cell 4
# Load and validate LogicRAG evidence files.
# This notebook requires final_evidence_memory and does not rebuild retrieval evidence.

def load_json_list(json_path: Path, dataset_name: str) -> List[Dict[str, Any]]:
    """Load a JSON file whose root must be a list."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"{dataset_name}: JSON root must be a list.")

    return data

def validate_evidence_record(record: Dict[str, Any], dataset_name: str, index: int) -> None:
    """Validate one LogicRAG evidence record for final-answer-only generation."""
    required_keys = {"type", "question", "answer", "final_evidence_memory"}

    if not isinstance(record, dict):
        raise ValueError(f"{dataset_name}: record {index} is not a dictionary.")

    missing = required_keys - set(record.keys())
    if missing:
        raise ValueError(f"{dataset_name}: record {index} is missing keys: {missing}")

    if not isinstance(record["question"], str) or not record["question"].strip():
        raise ValueError(f"{dataset_name}: record {index} has an empty question.")

    if not isinstance(record["final_evidence_memory"], str) or not record["final_evidence_memory"].strip():
        raise ValueError(
            f"{dataset_name}: record {index} has empty final_evidence_memory."
        )

def load_and_validate_dataset(dataset_name: str, evidence_path: Path) -> List[Dict[str, Any]]:
    """Load and validate one dataset evidence file."""
    records = load_json_list(evidence_path, dataset_name)

    if len(records) != EXPECTED_NUM_RECORDS_PER_DATASET:
        print(
            f"Warning: {dataset_name} has {len(records)} records, "
            f"expected {EXPECTED_NUM_RECORDS_PER_DATASET}."
        )

    for i, rec in enumerate(records):
        validate_evidence_record(rec, dataset_name, i)

    print("=" * 100)
    print(f"{dataset_name}: validation passed.")
    print("Number of records:", len(records))
    print("First question:", records[0]["question"])
    print("First answer / GT:", records[0]["answer"])
    print("First final_evidence_memory preview:")
    print(records[0]["final_evidence_memory"][:1000])

    return records

evidence_data = {}

for dataset_name, cfg in DATASETS.items():
    evidence_data[dataset_name] = load_and_validate_dataset(
        dataset_name=dataset_name,
        evidence_path=cfg["local_evidence_path"],
    )

hotpotqa: validation passed.
Number of records: 1000
First question: George Gershwin is an American Composer and Judith Weir is a composer from which country?
First answer / GT: a British composer
First final_evidence_memory preview:
George Gershwin was an American composer and pianist (1898–1937), renowned for works such as "Rhapsody in Blue" and "Porgy and Bess." Judith Weir is a British composer, born on 11 May 1954 in Cambridge, England, to Scottish parents. She is best known for her operas and theatrical works, including "The Black Spider" (1985), "A Night at the Chinese Opera" (1987), "The Vanishing Bridegroom" (1990), and "Blond Eckbert" (1994). In 2014, Weir was appointed Master of the Queen's Music, succeeding Sir Peter Maxwell Davies, becoming the first woman to hold the post. Her musical style is described as conservative yet modernist, drawing on medieval history and Scottish traditions, and she has received numerous accolades, including a CBE in 2005 and The Queen's Medal 

In [5]:
#cell 5
# Start GPT-OSS vLLM server.

def kill_process_tree(pid: int) -> None:
    """Kill a process and all child processes."""
    try:
        parent = psutil.Process(int(pid))
        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass
        parent.kill()
        parent.wait(timeout=10)
        print("Killed process tree:", pid)
    except Exception:
        pass

def find_nvidia_library_dirs() -> List[str]:
    """Find CUDA shared library directories installed by pip packages."""
    dirs = []

    for base in site.getsitepackages():
        pattern = os.path.join(base, "nvidia", "*", "lib")
        for d in glob.glob(pattern):
            if os.path.isdir(d):
                dirs.append(d)

    unique_dirs = []
    seen = set()

    for d in dirs:
        if d not in seen:
            unique_dirs.append(d)
            seen.add(d)

    return unique_dirs

if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(int(old_pid))

for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(3)

cmd = [
    "vllm", "serve", LLM_MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    "--generation-config", "vllm",
    "--trust-remote-code",

    "--dtype", "auto",
]

if KV_CACHE_DTYPE:
    cmd.extend(["--kv-cache-dtype", KV_CACHE_DTYPE])

if MAX_CUDAGRAPH_CAPTURE_SIZE:
    cmd.extend(["--max-cudagraph-capture-size", str(MAX_CUDAGRAPH_CAPTURE_SIZE)])

if not ENABLE_PREFIX_CACHING:
    cmd.append("--no-enable-prefix-caching")
else:
    cmd.append("--enable-prefix-caching")

server_env = os.environ.copy()

server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"
server_env["TORCH_CUDA_ARCH_LIST"] = "12.0"
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "1" if USE_FLASHINFER_SAMPLER else "0"

if DISABLE_FLASHINFER_COMPLETELY:
    server_env["VLLM_DISABLE_FLASHINFER"] = "1"

if ATTENTION_BACKEND:
    server_env["VLLM_ATTENTION_BACKEND"] = ATTENTION_BACKEND

if ADD_NVIDIA_PIP_LIBS_TO_LD_LIBRARY_PATH:
    nvidia_lib_dirs = find_nvidia_library_dirs()
    old_ld_path = server_env.get("LD_LIBRARY_PATH", "")
    merged_ld_path = ":".join(nvidia_lib_dirs + ([old_ld_path] if old_ld_path else []))
    server_env["LD_LIBRARY_PATH"] = merged_ld_path

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in [
    "VLLM_MAIN_CUDA_VERSION",
    "TORCH_CUDA_ARCH_LIST",
    "VLLM_USE_FLASHINFER_SAMPLER",
    "VLLM_DISABLE_FLASHINFER",
    "VLLM_ATTENTION_BACKEND",
    "LD_LIBRARY_PATH",
]:
    value = server_env.get(k)

    if k == "LD_LIBRARY_PATH" and value:
        print(f"{k}={value[:500]}{'...' if len(value) > 500 else ''}")
    else:
        print(f"{k}={value}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Command:
vllm serve openai/gpt-oss-120b --host 0.0.0.0 --port 8000 --max-model-len 131072 --gpu-memory-utilization 0.95 --max-num-seqs 1 --max-num-batched-tokens 8192 --generation-config vllm --trust-remote-code --dtype auto --kv-cache-dtype fp8 --max-cudagraph-capture-size 2048 --no-enable-prefix-caching

Important environment variables:
VLLM_MAIN_CUDA_VERSION=13.0
TORCH_CUDA_ARCH_LIST=12.0
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_DISABLE_FLASHINFER=None
VLLM_ATTENTION_BACKEND=None
LD_LIBRARY_PATH=/usr/local/lib/python3.12/dist-packages/nvidia/nccl/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cusparselt/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cufile/lib:/usr/local/lib/python3.12/dist-packages/nvidia/nvshmem/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cufft/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvcc/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cusolver/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvrtc/lib:/usr/local/lib/python3.12..

In [6]:
#cell 6
# Wait for vLLM server and create an OpenAI-compatible client.

import requests

def tail_log(path: Path, n: int = 80) -> str:
    """Read the last n log lines."""
    if not path.exists():
        return ""

    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False
SERVER_MODEL_ID = None

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()
    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=5)
        if h.status_code == 200:
            m = requests.get(f"{BASE_URL}/models", timeout=10)
            if m.status_code == 200:
                ready = True
                model_info = m.json()["data"][0]
                SERVER_MODEL_ID = model_info["id"]
                print("vLLM server is ready.")
                print("Model:", SERVER_MODEL_ID)
                print("Max model len:", model_info.get("max_model_len"))
                break
    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")
        recent = tail_log(SERVER_LOG_PATH, n=12)

        if recent.strip():
            print(recent)

        print("-" * 100)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

client = OpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    timeout=3600,
)

# Deterministic decoding keeps the final-answer-only stage close to the LogicRAG Qwen notebook.
LLM_SAMPLING_KWARGS = {
    "temperature": 0.0,
    "top_p": 1.0,
    "presence_penalty": 0.0,
}

# GPT-OSS reasoning is enabled internally, but only final message content is saved.
LLM_EXTRA_BODY = {
    "chat_template_kwargs": {
        "reasoning_effort": REASONING_EFFORT,
    },
    "include_reasoning": True,
}

llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True,
)

print("OpenAI-compatible client is ready.")
print("Server model id:", SERVER_MODEL_ID)
print("Reasoning effort:", REASONING_EFFORT)
print("Sampling kwargs:", LLM_SAMPLING_KWARGS)
print("Extra body:", LLM_EXTRA_BODY)
print("LLM tokenizer loaded.")

Waiting... 0s
----------------------------------------------------------------------------------------------------
Waiting... 60s
(EngineCore pid=5961) INFO 07-02 06:36:35 [parallel_state.py:1588] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.28.0.12:53987 backend=nccl
(EngineCore pid=5961) INFO 07-02 06:36:35 [parallel_state.py:1923] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0, EPLB rank N/A
(EngineCore pid=5961) INFO 07-02 06:36:35 [topk_topp_sampler.py:39] FlashInfer top-p/top-k sampling disabled via VLLM_USE_FLASHINFER_SAMPLER=0.
(EngineCore pid=5961) INFO 07-02 06:36:35 [gpu_model_runner.py:5159] Starting to load model openai/gpt-oss-120b...
(EngineCore pid=5961) Failed to get device capability: SM 12.x requires CUDA >= 12.9.
(EngineCore pid=5961) Failed to get device capability: SM 12.x requires CUDA >= 12.9.
(EngineCore pid=5961) INFO 07-02 06:36:37 [cuda.py:483] Using TRITON_ATTN attention backend out of pote

OpenAI-compatible client is ready.
Server model id: openai/gpt-oss-120b
Reasoning effort: high
Sampling kwargs: {'temperature': 0.0, 'top_p': 1.0, 'presence_penalty': 0.0}
Extra body: {'chat_template_kwargs': {'reasoning_effort': 'high'}, 'include_reasoning': True}
LLM tokenizer loaded.


In [7]:
#cell 7
# Helper functions for LogicRAG final-answer prompt, token truncation, cleanup, and JSON saving.

def atomic_save_json(obj: Any, path: Path) -> None:
    """Atomically save JSON to avoid corrupt partial files."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

    os.replace(tmp, path)

def load_json_if_exists(path: Path, default: Any) -> Any:
    """Load JSON if it exists; otherwise return default."""
    if not path.exists():
        return default

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def backup_file_if_exists(path: Path) -> Optional[Path]:
    """Create a timestamped backup of an existing file."""
    if not path.exists():
        return None

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    backup_path = path.with_name(path.stem + f".backup_{timestamp}" + path.suffix)
    shutil.copy2(path, backup_path)
    return backup_path

def strip_thinking_blocks(text: str) -> str:
    """Remove possible reasoning or special blocks from model output."""
    if not isinstance(text, str):
        return ""

    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = text.replace("<think>", "").replace("</think>", "")

    # Remove possible harmony-like special tokens if they appear in text.
    text = re.sub(r"<\|[^>]+?\|>", "", text)

    return text.strip()

def clean_final_answer(text: str) -> str:
    """Clean the final answer text without changing its meaning."""
    text = strip_thinking_blocks(text)
    text = text.replace("```json", "").replace("```text", "").replace("```", "").strip()
    text = re.sub(r"^\s*(ans|answer|final)\s*:\s*", "", text, flags=re.IGNORECASE).strip()
    text = text.strip().strip('"').strip("'").strip()
    return text

def count_llm_tokens(text: str) -> int:
    """Count tokens using the GPT-OSS tokenizer."""
    return len(llm_tokenizer.encode(str(text), add_special_tokens=False))

def truncate_to_tokens(text: str, max_tokens: int) -> str:
    """Truncate text to a token budget while preserving the beginning and the end."""
    if not isinstance(text, str):
        text = str(text)

    ids = llm_tokenizer.encode(text, add_special_tokens=False)

    if len(ids) <= max_tokens:
        return text

    if max_tokens <= 128:
        return llm_tokenizer.decode(ids[:max_tokens], skip_special_tokens=True)

    head_tokens = min(1024, max_tokens // 4)
    tail_tokens = max_tokens - head_tokens
    kept = ids[:head_tokens] + ids[-tail_tokens:]

    return llm_tokenizer.decode(kept, skip_special_tokens=True)

def build_logicrag_final_prompt(question: str, final_evidence_memory: str) -> Dict[str, Any]:
    """
    Build the exact final-answer prompt structure used by the LogicRAG Qwen notebook.
    Only question and final_evidence_memory are included.
    """
    info_summary = truncate_to_tokens(final_evidence_memory, MEMORY_MAX_TOKENS)

    prompt = f"""You must give ONLY the direct answer in the most concise way possible. DO NOT explain or provide any additional context.
If the answer is a simple yes/no, just say "Yes." or "No."
If the answer is a name, just give the name.
If the answer is a date, just give the date.
If the answer is a number, just give the number.
If the answer requires a brief phrase, make it as concise as possible.

Question: {question}

Information Summary:
{info_summary}

Remember: Be concise - give ONLY the essential answer, nothing more.
Ans: """

    original_tokens = count_llm_tokens(prompt)
    final_prompt = truncate_to_tokens(prompt, MAX_INPUT_TOKENS)
    final_tokens = count_llm_tokens(final_prompt)

    return {
        "prompt": final_prompt,
        "input_tokens": final_tokens,
        "was_truncated": final_tokens < original_tokens,
        "original_input_tokens": original_tokens,
        "memory_tokens": count_llm_tokens(info_summary),
    }

def extract_chat_message_content(completion: Any) -> str:
    """Extract only the final assistant message content from an OpenAI-compatible chat completion."""
    message = completion.choices[0].message

    content = getattr(message, "content", None)

    if content is None and isinstance(message, dict):
        content = message.get("content")

    # Reasoning content is intentionally not saved.
    if content is None:
        content = ""

    return str(content)

def llm_chat_final(prompt: str, max_retries: int = 3) -> str:
    """
    Call the local GPT-OSS vLLM server using the same final-answer system behavior
    as the LogicRAG Qwen notebook.
    """
    prompt = truncate_to_tokens(prompt, MAX_INPUT_TOKENS)

    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant. Do not think step by step. Do not reveal hidden reasoning.",
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            completion = client.chat.completions.create(
                model=SERVER_MODEL_ID or LLM_MODEL_NAME,
                messages=messages,
                max_tokens=ANSWER_MAX_TOKENS,
                **LLM_SAMPLING_KWARGS,
                extra_body=LLM_EXTRA_BODY,
            )

            raw_text = extract_chat_message_content(completion)
            cleaned = clean_final_answer(raw_text)

            if cleaned:
                return cleaned

            print(f"Warning: empty final answer on attempt {attempt}/{max_retries}. Retrying...")
            time.sleep(2 * attempt)

        except Exception as e:
            last_error = e
            print(f"LLM call failed on attempt {attempt}/{max_retries}: {repr(e)}")
            time.sleep(2 * attempt)

    if last_error is not None:
        raise RuntimeError(f"LLM call failed after {max_retries} attempts: {repr(last_error)}")

    return ""

def validate_answer_record(record: Dict[str, Any], dataset_name: str, index: int) -> None:
    """Validate one final answer output record."""
    required_keys = {"type", "question", "gt", "response"}

    if set(record.keys()) != required_keys:
        raise ValueError(
            f"{dataset_name}: answer record {index} must have exactly {required_keys}, "
            f"but found {set(record.keys())}"
        )

print("Final-answer helper functions are ready.")

Final-answer helper functions are ready.


In [8]:
#cell 8
# Show one prompt preview.
# This is only for checking that the prompt uses question + final_evidence_memory.

sample_dataset = "hotpotqa"
sample_record = evidence_data[sample_dataset][0]

sample_prompt_info = build_logicrag_final_prompt(
    question=sample_record["question"],
    final_evidence_memory=sample_record["final_evidence_memory"],
)

print("Sample dataset:", sample_dataset)
print("Sample input tokens:", sample_prompt_info["input_tokens"])
print("Sample memory tokens:", sample_prompt_info["memory_tokens"])
print("Sample was truncated:", sample_prompt_info["was_truncated"])

print("\nSample prompt preview:")
print(sample_prompt_info["prompt"][:2500])

Sample dataset: hotpotqa
Sample input tokens: 330
Sample memory tokens: 197
Sample was truncated: False

Sample prompt preview:
You must give ONLY the direct answer in the most concise way possible. DO NOT explain or provide any additional context.
If the answer is a simple yes/no, just say "Yes." or "No."
If the answer is a name, just give the name.
If the answer is a date, just give the date.
If the answer is a number, just give the number.
If the answer requires a brief phrase, make it as concise as possible.

Question: George Gershwin is an American Composer and Judith Weir is a composer from which country?

Information Summary:
George Gershwin was an American composer and pianist (1898–1937), renowned for works such as "Rhapsody in Blue" and "Porgy and Bess." Judith Weir is a British composer, born on 11 May 1954 in Cambridge, England, to Scottish parents. She is best known for her operas and theatrical works, including "The Black Spider" (1985), "A Night at the Chinese Opera" (19

In [9]:
#cell 9
# Test one GPT-OSS final-answer call before running the full datasets.
# The GT is printed only for human checking and is not sent to the model.

test_dataset = "hotpotqa"
test_record = evidence_data[test_dataset][0]

test_prompt_info = build_logicrag_final_prompt(
    question=test_record["question"],
    final_evidence_memory=test_record["final_evidence_memory"],
)

print("Test dataset:", test_dataset)
print("Test question:", test_record["question"])
print("Test GT answer:", test_record["answer"])
print("Note: GT is printed only for human checking and is not sent to the LLM.")
print("Test input tokens:", test_prompt_info["input_tokens"])
print("Test memory tokens:", test_prompt_info["memory_tokens"])
print("Test was truncated:", test_prompt_info["was_truncated"])

test_response = llm_chat_final(test_prompt_info["prompt"])

print("Test GPT-OSS response:", test_response)

Test dataset: hotpotqa
Test question: George Gershwin is an American Composer and Judith Weir is a composer from which country?
Test GT answer: a British composer
Note: GT is printed only for human checking and is not sent to the LLM.
Test input tokens: 330
Test memory tokens: 197
Test was truncated: False
Test GPT-OSS response: United Kingdom


In [10]:
#cell 10
# Dataset processing function.
# This performs only the final LogicRAG answer-generation stage.
# It does not retrieve, rerank, summarize, or modify evidence files.

def get_processing_slice(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Return the selected processing slice."""
    end = ANSWER_END_INDEX if ANSWER_END_INDEX is not None else len(records)
    return records[ANSWER_START_INDEX:end]

def load_existing_answers(answer_path: Path) -> List[Dict[str, Any]]:
    """Load existing answers for resume mode."""
    if not answer_path.exists():
        return []

    with open(answer_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"Existing answer file must contain a list: {answer_path}")

    return data

def load_or_initialize_answer_records(
    dataset_name: str,
    selected_records: List[Dict[str, Any]],
    answer_path: Path,
) -> List[Dict[str, Any]]:
    """Load existing answer records only if they match the selected evidence prefix."""
    if not RESUME_IF_EXISTS or not answer_path.exists():
        return []

    existing = load_existing_answers(answer_path)

    can_resume = True

    if len(existing) > len(selected_records):
        can_resume = False
    else:
        for i, old in enumerate(existing):
            if old.get("question") != selected_records[i].get("question"):
                can_resume = False
                break

    if can_resume:
        for i, rec in enumerate(existing):
            validate_answer_record(rec, dataset_name, i)

        print(f"{dataset_name}: resuming from {len(existing)} existing answers.")
        return existing

    backup_path = backup_file_if_exists(answer_path)
    print(f"{dataset_name}: existing GPT-OSS answer file does not match current data.")
    print("Starting over.")
    if backup_path is not None:
        print("Backup:", backup_path)

    return []

def process_dataset(dataset_name: str) -> Dict[str, Any]:
    """
    Process one dataset independently.

    Important:
    - Only question and final_evidence_memory are given to the LLM.
    - answer, supports, evidence_chunk, and any other fields are not included in the prompt.
    - The saved JSON contains exactly: type, question, gt, response.
    """
    cfg = DATASETS[dataset_name]
    records = evidence_data[dataset_name]
    answer_path = cfg["answer_path"]

    print("=" * 100)
    print(f"Starting dataset: {dataset_name}")
    print("Answer path:", answer_path)

    selected_records = get_processing_slice(records)

    if not selected_records:
        raise ValueError(f"{dataset_name}: selected record slice is empty.")

    answer_records = load_or_initialize_answer_records(
        dataset_name=dataset_name,
        selected_records=selected_records,
        answer_path=answer_path,
    )

    start_time = time.time()

    input_token_counts = []
    memory_token_counts = []
    truncation_count = 0

    start_i = len(answer_records)

    for local_i in tqdm(
        range(start_i, len(selected_records)),
        desc=f"GPT-OSS final answers: {dataset_name}",
        dynamic_ncols=True,
    ):
        rec = selected_records[local_i]

        question = rec["question"]
        final_evidence_memory = rec["final_evidence_memory"]

        prompt_info = build_logicrag_final_prompt(
            question=question,
            final_evidence_memory=final_evidence_memory,
        )

        input_token_counts.append(prompt_info["input_tokens"])
        memory_token_counts.append(prompt_info["memory_tokens"])

        if prompt_info.get("was_truncated", False):
            truncation_count += 1

        try:
            response = llm_chat_final(prompt_info["prompt"])
        except Exception as e:
            print("=" * 100)
            print(f"Error on dataset={dataset_name}, local_index={local_i}")
            print("Question:", question)
            print("Exception:", e)
            print(traceback.format_exc())
            response = ""

        output_record = {
            "type": rec.get("type", ""),
            "question": question,
            "gt": rec.get("answer", ""),
            "response": response,
        }

        validate_answer_record(output_record, dataset_name, local_i)
        answer_records.append(output_record)

        if len(answer_records) % SAVE_EVERY_N == 0:
            atomic_save_json(answer_records, answer_path)

        if CLEAR_CACHE_EVERY_N and len(answer_records) % CLEAR_CACHE_EVERY_N == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    atomic_save_json(answer_records, answer_path)

    elapsed = time.time() - start_time
    saved = load_existing_answers(answer_path)

    if len(saved) != len(selected_records):
        raise ValueError(
            f"{dataset_name}: saved {len(saved)} answers, "
            f"but expected {len(selected_records)}."
        )

    for i, out_rec in enumerate(saved):
        validate_answer_record(out_rec, dataset_name, i)

    summary = {
        "dataset": dataset_name,
        "num_input_records": len(records),
        "num_processed_records": len(saved),
        "answer_path": str(answer_path),
        "elapsed_seconds": elapsed,
        "num_truncated_prompts": truncation_count,
        "max_input_tokens_seen": max(input_token_counts) if input_token_counts else None,
        "mean_input_tokens_seen": float(sum(input_token_counts) / len(input_token_counts)) if input_token_counts else None,
        "max_memory_tokens_seen": max(memory_token_counts) if memory_token_counts else None,
        "mean_memory_tokens_seen": float(sum(memory_token_counts) / len(memory_token_counts)) if memory_token_counts else None,
    }

    print("=" * 100)
    print(f"{dataset_name}: done.")
    print("Saved to:", answer_path)
    print("Processed records:", len(saved))
    print("Elapsed seconds:", elapsed)
    print("Truncated prompts:", truncation_count)
    print("First output record:")
    print(json.dumps(saved[0], ensure_ascii=False, indent=2))

    return summary

print("Dataset processing function is ready.")

Dataset processing function is ready.


In [11]:
#cell 11
# Run HotpotQA separately.

hotpotqa_gpt_oss_summary = process_dataset("hotpotqa")
hotpotqa_gpt_oss_summary

Starting dataset: hotpotqa
Answer path: /content/drive/MyDrive/final_project/logicRAG/answer/hotpotqa_gpt_oss_120b_answers.json


GPT-OSS final answers: hotpotqa:   0%|          | 0/1000 [00:00<?, ?it/s]

hotpotqa: done.
Saved to: /content/drive/MyDrive/final_project/logicRAG/answer/hotpotqa_gpt_oss_120b_answers.json
Processed records: 1000
Elapsed seconds: 589.1211800575256
Truncated prompts: 0
First output record:
{
  "type": "comparison",
  "question": "George Gershwin is an American Composer and Judith Weir is a composer from which country?",
  "gt": "a British composer",
  "response": "United Kingdom"
}


{'dataset': 'hotpotqa',
 'num_input_records': 1000,
 'num_processed_records': 1000,
 'answer_path': '/content/drive/MyDrive/final_project/logicRAG/answer/hotpotqa_gpt_oss_120b_answers.json',
 'elapsed_seconds': 589.1211800575256,
 'num_truncated_prompts': 0,
 'max_input_tokens_seen': 868,
 'mean_input_tokens_seen': 261.295,
 'max_memory_tokens_seen': 737,
 'mean_memory_tokens_seen': 127.156}

In [12]:
#cell 12
# Run 2WikiMultiHopQA separately.

wikimultihopqa_gpt_oss_summary = process_dataset("2wikimultihopqa")
wikimultihopqa_gpt_oss_summary

Starting dataset: 2wikimultihopqa
Answer path: /content/drive/MyDrive/final_project/logicRAG/answer/2wikimultihopqa_gpt_oss_120b_answers.json


GPT-OSS final answers: 2wikimultihopqa:   0%|          | 0/1000 [00:00<?, ?it/s]

2wikimultihopqa: done.
Saved to: /content/drive/MyDrive/final_project/logicRAG/answer/2wikimultihopqa_gpt_oss_120b_answers.json
Processed records: 1000
Elapsed seconds: 551.9234783649445
Truncated prompts: 0
First output record:
{
  "type": "bridge_comparison",
  "question": "Do both films, Shadows Of The Metropolis and Jaider, Der Einsame Jäger, have the directors who are from the same country?",
  "gt": "yes",
  "response": "Yes."
}


{'dataset': '2wikimultihopqa',
 'num_input_records': 1000,
 'num_processed_records': 1000,
 'answer_path': '/content/drive/MyDrive/final_project/logicRAG/answer/2wikimultihopqa_gpt_oss_120b_answers.json',
 'elapsed_seconds': 551.9234783649445,
 'num_truncated_prompts': 0,
 'max_input_tokens_seen': 907,
 'mean_input_tokens_seen': 237.482,
 'max_memory_tokens_seen': 771,
 'mean_memory_tokens_seen': 106.962}

In [13]:
#cell 13
# Verify final GPT-OSS answer files.

def verify_final_answer_file(dataset_name: str, answer_path: Path) -> None:
    """Verify final answer JSON format."""
    records = load_existing_answers(answer_path)

    if not records:
        raise ValueError(f"{dataset_name}: answer file is empty: {answer_path}")

    for i, rec in enumerate(records):
        validate_answer_record(rec, dataset_name, i)

    print("=" * 100)
    print(f"{dataset_name}: final answer file verified.")
    print("Path:", answer_path)
    print("Number of records:", len(records))
    print("First record keys:", list(records[0].keys()))
    print("First question:", records[0]["question"])
    print("First GT:", records[0]["gt"])
    print("First response:", records[0]["response"])

for dataset_name, cfg in DATASETS.items():
    verify_final_answer_file(dataset_name, cfg["answer_path"])

print("\nAnswer folder contents:")
for item in sorted(DRIVE_ANSWER_DIR.iterdir()):
    print(" -", item.name)

hotpotqa: final answer file verified.
Path: /content/drive/MyDrive/final_project/logicRAG/answer/hotpotqa_gpt_oss_120b_answers.json
Number of records: 1000
First record keys: ['type', 'question', 'gt', 'response']
First question: George Gershwin is an American Composer and Judith Weir is a composer from which country?
First GT: a British composer
First response: United Kingdom
2wikimultihopqa: final answer file verified.
Path: /content/drive/MyDrive/final_project/logicRAG/answer/2wikimultihopqa_gpt_oss_120b_answers.json
Number of records: 1000
First record keys: ['type', 'question', 'gt', 'response']
First question: Do both films, Shadows Of The Metropolis and Jaider, Der Einsame Jäger, have the directors who are from the same country?
First GT: yes
First response: Yes.

Answer folder contents:
 - 2wikimultihopqa_gpt_oss_120b_answers.json
 - 2wikimultihopqa_qwen3.5_answers.json
 - hotpotqa_gpt_oss_120b_answers.json
 - hotpotqa_qwen3.5_answers.json
